In [ ]:
%%configure -f
{
    "jars": ["https://repo1.maven.org/maven2/org/datasyslab/geotools-wrapper/1.9.1-33.5/geotools-wrapper-1.9.1-33.5.jar", "https://repo1.maven.org/maven2/org/apache/sedona/sedona-spark-shaded-4.1_2.13/1.9.1/sedona-spark-shaded-4.1_2.13-1.9.1.jar"]
}

In [ ]:
%run manage_tables

In [ ]:
from sedona.spark import SedonaContext

In [ ]:
TAXI_ZONES_SHAPEFILE_FOLDER_PATH = "Files/NYC_Taxi/taxi_zones/"

In [ ]:
def create_sedona_context(spark):
    return SedonaContext.create(spark)

def read_shapefile(sedona, file_path):
    return sedona.read.format("org.apache.sedona.sql.datasources.shapefile.ShapefileDataSource").load(file_path)

def transform_geometry_to_coordinates(df):
    return df.selectExpr("*", "ST_Transform(geometry, 'EPSG:2263', 'EPSG:4326') AS new_geometry")

def load_taxi_zones(sedona, shapefile_folder_path):
    zones_df = read_shapefile(sedona, shapefile_folder_path)
    zones_df = transform_geometry_to_coordinates(zones_df)
    create_temp_view(zones_df, "zones")

def prepare_spatial_processing(spark, shapefile_folder_path):
    spark.conf.set("sedona.join.autoBroadcastJoinThreshold", "-1")
    sedona = create_sedona_context(spark)
    load_taxi_zones(sedona, shapefile_folder_path)
    return sedona

In [ ]:
def add_zone_from_coordinates(df, sedona, latitude_column, longitude_column):
    create_temp_view(df, "df_table")
    return sedona.sql(f"""
        SELECT
            df_table.*,
            zones.LocationID AS location_id
        FROM df_table
        LEFT JOIN zones
        ON ST_Contains(zones.new_geometry, ST_Point(df_table.{longitude_column}, df_table.{latitude_column}))
    """)

In [ ]:
def add_zone_from_coordinates_or_location(df, sedona):
    create_temp_view(df, "trips")
    return sedona.sql("""
        SELECT
            trips.pickup_datetime,
            trips.dropoff_datetime,
            trips.passenger_count,
            trips.trip_distance,
            COALESCE(trips.pickup_location_id, pickup_zone.LocationID) AS pickup_location_id,
            COALESCE(trips.dropoff_location_id, dropoff_zone.LocationID) AS dropoff_location_id,
            trips.fare_amount,
            trips.total_amount,
            trips.tip_amount,
            trips.pickup_longitude,
            trips.pickup_latitude,
            trips.dropoff_longitude,
            trips.dropoff_latitude
        FROM trips
        LEFT JOIN zones AS pickup_zone 
        ON trips.pickup_location_id IS NULL
            AND ST_Contains(pickup_zone.new_geometry, ST_Point(trips.pickup_longitude, trips.pickup_latitude))
        LEFT JOIN zones AS dropoff_zone
        ON trips.dropoff_location_id IS NULL
            AND ST_Contains(dropoff_zone.new_geometry, ST_Point(trips.dropoff_longitude, trips.dropoff_latitude))
    """)